In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

import polars as pl
from polars import col, lit, when
import re

import sys
import os

root_path = os.path.abspath(os.path.join(os.getcwd(), '..'))
if root_path not in sys.path:
    sys.path.append(root_path)

from scripts.feature_calculation import build_processed_dataset
from scripts.train_eval_model import train_baseline

### Creating train dataframe with new features

При считывании создаем колонку, по которой можно отличить train от pretrain и приводим данные к одинаковой схеме

In [3]:
train_1 = pl.scan_parquet('../../data/train_part_1.parquet')
train_1 = train_1.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_1 = pl.scan_parquet('../../data/pretrain_part_1.parquet')
pretrain_1 = pretrain_1.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

train_2 = pl.scan_parquet('../../data/train_part_2.parquet')
train_2 = train_2.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_2 = pl.scan_parquet('../../data/pretrain_part_2.parquet')
pretrain_2 = pretrain_2.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

train_3 = pl.scan_parquet('../../data/train_part_3.parquet')
train_3 = train_3.with_columns(
    pl.lit(1).alias('is_train')
)
pretrain_3 = pl.scan_parquet('../../data/pretrain_part_3.parquet')
pretrain_3 = pretrain_3.with_columns(
    pl.col('session_id').cast(pl.Int64), # schema mismatch in session_id
    pl.lit(0).alias('is_train')
)

pretest = pl.scan_parquet('../../data/pretest.parquet')
pretest = pretest.with_columns(
    pl.lit(0).alias('is_train')
)
test = pl.scan_parquet('../../data/test.parquet')
test = test.with_columns(
    pl.lit(1).alias('is_train')
)

labels = pl.scan_parquet('../../data/train_labels.parquet')

In [4]:
full_train = pl.concat([pretrain_1, pretrain_2, pretrain_3, 
                        train_1, train_2, train_3, pretest, test], how='vertical') 

In [5]:
build_processed_dataset(full_train)

  100,000 customers -> 50 partitions x ~2,000 customers each
[██████████████████████████████] 100.0%  part 50/50  (1,851,217 train rows)  elapsed 18s  ETA 0s              

Done. 93,960,644 total train rows written across 50 files in '../data_processed/'.


Посмотрим что получилось

In [6]:
example_data = pl.read_parquet('../data_processed/part_0000.parquet')
example_data.head()

customer_id,event_id,event_dttm,event_type_nm,event_desc,channel_indicator_type,channel_indicator_sub_type,operaton_amt,currency_iso_cd,mcc_code,pos_cd,accept_language,browser_language,timezone,session_id,operating_system_type,battery,device_system_version,screen_size,developer_tools,phone_voip_call_state,web_rdp_connection,compromised,is_train,hour,day_of_week,day_of_month,week_of_year,hour_of_day,is_weekend,is_night,is_working_hour,minutes_from_midnight,log_amount,amount_abs,amount_round_100,amount_round_1000,…,global_event_desc_freq,circadian_deviation_score,channel_shift_score,velocity_change_flag,time_gap_variance_30d,new_device_flag,new_mcc_flag,new_channel_flag,merchant_entropy_user,browser_language_mismatch,new_device_and_night_flag,rdp_and_large_amount_flag,amount_zscore_given_channel,amount_zscore_given_mcc,amount_zscore_given_device,tx_time_zscore_given_user,amount_zscore_channel,amount_zscore_mcc,global_combination_freq,session_first_tx_flag,language_change_flag,os_change_flag,timezone_change_flag,rapid_sequence_flag,suspicious_env_flag,mcc_rare_global_flag,rare_combination_flag,device_change_and_large_amount_flag,voip_and_new_mcc_flag,compromised_and_high_amount_flag,session_first_tx_large_flag,geo_jump_proxy,device_entropy_ratio,language_mismatch,timezone_mismatch,merchant_last_seen_days,mcc_last_seen_days
i64,i64,datetime[μs],i32,i32,i32,i32,f32,i32,str,i32,str,str,i32,i64,i32,str,str,str,str,i32,i32,str,i32,i8,i8,i8,i8,i8,i8,i8,i8,i8,f32,f32,i8,i8,…,u32,f32,i8,i8,f32,i8,i8,i8,f32,i8,i8,i8,f32,f32,f32,f32,f32,f32,u32,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i8,i32,f32,i8,i8,i32,i32
123123123124290,123939168233221,2024-10-01 00:03:20,7,56,3,4,null,null,null,null,"""ru""",null,3,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,3,0.0,0.0,1,1,…,318,7.106918,0,0,1103.891479,0,0,0,0.0,0,0,0,1.3631e6,2.2602e6,1.6798e6,4.120892,-0.015369,-0.006747,499644,0,0,0,0,0,0,1,0,0,0,0,0,0,0.018868,1,0,0,0
123140302994545,123861859323955,2024-10-01 00:03:20,7,56,4,15,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,3,0.0,0.0,1,1,…,579,7.111576,0,1,1188.375732,0,0,0,0.0,0,0,0,1.0889e6,2.2602e6,1.6798e6,5.034358,-0.012839,-0.006747,2104642,0,0,0,0,0,0,1,0,0,0,0,0,0,0.007407,0,1,0,0
123131713060513,126533330625689,2024-10-01 00:05:46,14,75,6,5,23267.0,0,"""4""",3,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,5,10.054834,23267.0,0,0,…,268,6.835327,1,0,939.009521,0,0,0,0.11249,0,0,0,349331.46875,65604.1875,1.6798e6,5.683391,-0.125443,-0.326275,696868,0,0,0,0,0,0,0,0,0,0,0,0,0,0.01,0,1,0,1
123123123126335,125107399907706,2024-10-01 00:07:13,14,75,6,5,22498.0,0,null,3,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,7,10.021226,22498.0,0,0,…,925,8.965702,1,0,506.841522,0,0,0,0.0,0,0,0,349331.46875,2.2602e6,1.6798e6,5.58534,-0.125739,-0.006679,696868,0,0,0,0,0,0,1,0,0,0,0,0,0,0.004464,0,1,0,0
123131713060293,126181143683685,2024-10-01 00:09:25,7,56,3,4,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,0,2,1,40,0,0,1,0,9,0.0,0.0,1,1,…,534,6.899468,1,0,647.843628,0,0,0,0.0,0,0,0,1.3631e6,2.2602e6,1.6798e6,4.94799,-0.015369,-0.006747,499644,0,0,0,0,0,0,1,0,0,0,0,0,0,0.008547,0,1,0,0


In [7]:
print(full_train.filter(pl.col('is_train') == 1).select(pl.col('event_dttm').min()).collect())
print(full_train.filter(pl.col('is_train') == 1).select(pl.col('event_dttm').max()).collect())

shape: (1, 1)
┌─────────────────────┐
│ event_dttm          │
│ ---                 │
│ str                 │
╞═════════════════════╡
│ 2024-10-01 00:00:03 │
└─────────────────────┘
shape: (1, 1)
┌─────────────────────┐
│ event_dttm          │
│ ---                 │
│ str                 │
╞═════════════════════╡
│ 2025-08-09 23:58:25 │
└─────────────────────┘


### Training baseline on engineered features

In [3]:
train_baseline()

Found 50 parquet partitions in '../data_processed'

  Cache miss — missing: ['memmap_meta.json', 'X_train.npy', 'y_train.npy', 'X_val.npy', 'y_val.npy', 'is_labeled_val.npy']
Loading labels …
  87,514 labelled rows  |  51,438 positives  (58.777%)

Pre-scan: counting train/val rows for memmap allocation …
  [███████████████████████████████████] 100.0%  50/50  ETA 0s  train 65,951,461 | val 27,097,763          
  train rows: 65,951,461  |  val rows: 27,097,763

Detecting feature count from first chunk …
  161 feature columns detected.

Allocating memmap files …
  X_train.npy : 65,951,461 × 161  ≈ 42.5 GB on disk
  X_val.npy   : 27,097,763 × 161  ≈ 17.5 GB on disk

Writing chunks into memmap files …
  [███████████████████████████████████] 100.0%  50/50  ETA 0s  written train 65,951,461 | val 27,097,763             

  Train : 65,951,461 rows  | 41,801 positives (0.0634%)
  Val   : 27,097,763 rows  | 15,515 positives (0.0573%)

  Metadata saved → ..\data_splits\memmap_meta.json
  Feature c

In [ ]:
# Final PR-AUC    : 0.001098

In [4]:
sub = pd.read_csv('submission.csv')
sub.shape

(911420, 2)

In [5]:
esub = pd.read_csv('../../data/sample_submit.csv')
esub.shape

(633683, 2)

In [6]:
sub.event_id.unique().shape

(633683,)

In [7]:
sub = sub.drop_duplicates(subset=['event_id'])

In [8]:
sub.shape

(633683, 2)

In [10]:
sub.to_csv('submission_1.csv', index=False)

In [11]:
sub = pd.read_csv('submission_1.csv')
sub.head()

,event_id,predict
0,125390866897300,0.002353
1,126189731373139,0.011547
2,125081630705881,0.001674
3,125262020316273,0.001105
4,125682927274458,0.007236
